# Data Science Project 

## Research Question : - Does the adoption of electric cars reduce NOx emissions from road transport ? 

### By Alina Loacker, Amanda Loeb and Max Laurent

#### Under the supervision of Dr. Edoardo Chiarotti and Dr. Quentin Gallea

Midterm : Data Science & Causal Inference for Sustainability

Overview : 

1. Our Research Question: 
- Does the adoption of electric cars reduce NOx emissions from road transport? 

This question is causal and uses three variables, as recommended in the instructions:
Outcome variable: NOx emissions from transport (kg per municipality per year).
- Main explanatory variable: stock (or share) of electric cars per municipality per year.
- Heterogeneity variable: population density (or degree of urbanization).


2. Why does it matter? 

Air pollution is one of the most important health and environmental issues worldwide. The Global Burden of Disease study estimated that it caused about 3.4 million premature deaths in 2017, and the WHO identifies it as a risk factor for cardiovascular and respiratory diseases such as stroke, heart disease and lung cancer. NOx (nitrogen monoxide and nitrogen dioxide) are toxic gases formed during combustion at high temperature. They harm health and also contribute to ground-level ozone and soil acidification. 

Transport is a major source. In Sweden in 2019, transport accounted for about 43% of NOx emissions, road traffic for about 85% of that, and passenger cars for about 55% of road traffic emissions. Public policies promoting electric vehicles (for instance the Swedish "bonus-malus" system introduced in 2018) are mainly motivated by CO2 reduction. If electric cars also improve air quality, this is a co-benefit that strengthens the case for subsidizing them beyond what climate considerations alone would demonstrate.


3. Literature review (short)

- Ferrero et al. (2016) simulate emission scenarios near a highway in Milan. Replacing 50% of combustion vehicles with pure electric ones reduces NO2 by 5.5% and NO by 16%, but low replacement rates show little effect. Limitation: simulation, short period, very local area.
- Holland et al. (2016) study the USA and show that benefits depend strongly on local factors, mainly on how electricity is produced. Benefits are large in Los Angeles but negative in the Midwest. Limitation: data from 2010-2012 only.
- Choma et al. (2020) use an epidemiological model and find positive health benefits in every US metropolitan area, with larger estimates than Holland et al. Limitation: model-based, not an observational causal estimate.
- Andersson & Breuer (2021) use a panel of 290 Swedish municipalities (2010-2019) and OLS with municipality and year fixed effects. They find an elasticity of about -0.03 (significant at 1%). Limitations: NOx covers the whole transport sector, many omitted variables, missing values coded as 0, and fragile robustness.



4. Variables


| Variable | Role | Source |
|---|---|---|
| NOx emissions, "Road transport: Passenger cars" (kt/year) | Outcome | EEA air pollutant emissions inventory (cleaned by Max) |
| NOx emissions, national total (kt/year) | Context / share of cars in total NOx | EEA (Max) |
| Stock of pure electric cars (`mot_nrg` = ELC) | Main explanatory variable | Eurostat `road_eqs_carpda` (cleaned by Alina) |
| Total stock of passenger cars (`mot_nrg` = TOTAL) | Control / denominator of the EV share | Eurostat (Alina) |
| Population density / population | Heterogeneity and control | To be added (Eurostat / World Bank) |

Transformations : The stock of electric cars grows exponentially (about +973% between 2010 and 2019 nationally), and all variables are strongly right-skewed. We log-transform the variables so that the relationship is closer to linear and the coefficient can be read as an elasticity (a percentage change in NOx for a 1% change in electric cars). We also consider using electric cars **per capita** instead of raw stocks so that large cities do not dominate the analysis.

Limitations of the variables**
- Emissions are inventory estimates (modelled, "fuel sold" basis), not directly measured, so part of the link with the fleet is mechanical.
- Only pure electric cars (ELC) are counted: hybrids and plug-in hybrids are not included.
- Eurostat's `leg_form` (TOTAL / natural person / legal person) must be filtered to TOTAL, otherwise cars are counted several times.
- Aggregates in the EEA file (EEA32, EU27) must be dropped, and country names harmonized between the two sources.
- Country-level data hides local variation (cities vs. rural areas).

5. Step 1: Sample

The sample is a country-year panel of European countries, merged from Max's NOx data and Alina's Eurostat data (inner join on country and year).

- Period: starts in 2016 (first year with usable Eurostat car data across countries); NOx data from the EEA goes back to 2000 and is used for context plots.
- Why not end before 2020 as originally planned: the data stops later, so we keep 2020-2021 but run a robustness check without them (Covid-19 distorted traffic).
- Cleaning: keep NOx only, sector "Road transport: Passenger cars", unit kt; keep `leg_form == TOTAL`; drop EEA32/EU27; check duplicates, missing values, impossible values and consistent units.
- Cases: Norway and Sweden are highlighted as the main illustrative countries (Norway is the EV outlier).

We describe the sample with a summary statistics table (number of observations, mean, standard deviation, min, max) for NOx (kt), EV stock, total cars, EV share (%) and NOx per car.

6. Step 2: Univariate analysis

- **NOx (outcome):** histogram in levels (strongly right-skewed because of country size) and in logs (more symmetric). Time series of passenger-car NOx for Norway and Sweden (from Max's work): a declining trend over time.
- **Share of road transport in national NOx:** Max's merged dataset shows the road-transport share over time (e.g. about 7-9% for Norway in 2000-2009), useful to show how much room electric cars have to matter.
- **EV stock (explanatory variable):** from Alina's Eurostat data, near zero in early years then exponential growth; very right-skewed in levels, so we use logs and EV share.
- **Heterogeneity variable:** distribution of population density across countries (to be added).

7. Step 3: Bivariate analysis

- Scatter plot of log NOx vs. log EV stock (pooled), and the same after removing country means (within-country), to see the relationship with and without fixed effects.
- Correlation table between log NOx, EV share, total cars and density.
- Expected pitfall: the pooled relationship is driven by country size (big countries have both more EVs and more NOx), so the within-country relationship is the informative one. Both trends are also time-driven, so year effects are needed.

8. Step 4: Causality issues

We answer the three questions from the instructions.

Is there something else? (omitted variables)
- Wealth, education or environmental aspects affect both electric car purchases and emissions.
- Weather and temperature influence both emissions and electric vehicle range.
- Common trends such as stricter Euro emission standards (Euro 6, real-driving emissions tests) reduce NOx everywhere, regardless of electric cars. Year fixed effects help but do not remove every confounder.
- Fleet renewal: total fleet size and diesel share change at the same time as EV adoption.
- Measurement: inventory NOx is set from fleet composition, so the link with EV stock is partly mechanical.

Is it the reverse? (reverse causality)
- Countries or cities with worse air quality may adopt electric cars faster, or national policies such as (subsidies, low-emission zones, charging infrastructure) may drive both variables at once.

Can we extrapolate? (external validity)
- Electricity mix matters especially : Norway and Sweden have very low-carbon electricity, so benefits may be larger than in countries relying on fossil fuels (Holland et al.). Note that tailpipe NOx falls regardless; the mix matters mostly for power-plant NOx.
- Norway is an extreme outlier in EV share (we run a robustness check without it). Nordic countries have high incomes and strong environmental policy, so results may not apply elsewhere.

Possible strategies: country and year fixed effects with country-clustered standard errors, additional controls (fleet size, density, income), robustness (excluding 2020-2021 and Norway), and a quasi-experimental design such as difference-in-differences or an instrumental variable based on national EV policy reforms (e.g. the 2018 Swedish bonus-malus).

9. Presentation structure (max 8 slides)

1. **Research question and motivation:** health impact of NOx, role of transport, co-benefit argument.
2. **Literature and gap:** short narrative linking the four studies; our contribution (passenger cars, cross-country).
3. **Ingredients:** variables, sources (EEA, Eurostat), transformations, limitations.
4. **Sample:** sample selection, summary statistics table.
5. **Univariate analysis:** histograms, time series (Norway, Sweden), EV growth.
6. **Bivariate analysis:** scatter plots and correlations, with and without fixed effects.
7. **Causality issues:** omitted variables, reverse causality, extrapolation.
8. **Next steps:** chosen method and planned extensions up to the final report.
